<a href="https://colab.research.google.com/github/mariaceciliakv-ui/GenerationGITHUB/blob/main/Coleta_de_dados_meteorologicos.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [10]:
#Coleta de dados meteorologicos
#Bibliotecas
import pandas as pd
import requests
import time
import sys

endpoint_planilha = "https://docs.google.com/spreadsheets/d/1TSOTrJwH1ljjFD1seehdeFvi25-q_jfM8-f16Ssuhf4/export?format=csv"
endpoint_api = "https://api.open-meteo.com/v1/forecast"

def limpar_coordenada(valor):
    """Função para limpar e padronizar as coordenadas recebidas da planilha."""
    if pd.isna(valor):
        return None

    # Converte para string e limpa espaços
    texto = str(valor).strip()

    # Substitui vírgula por ponto (caso de Campinas: '22,9' -> '22.9')
    texto = texto.replace(',', '.')

    # Caso de pontos de milhar extras (ex: '-23.046.753' -> '-23.046753')
    if texto.count('.') > 1:
        partes = texto.split('.')
        # Junta tudo exceto a última parte com uma string vazia, e adiciona a última parte com o ponto decimal final
        texto = "".join(partes[:-1]) + "." + partes[-1]

    try:
        return float(texto)
    except ValueError:
        return None

#Coletar dados
try:
    dados_planilha = pd.read_csv(endpoint_planilha)
    print("-----Dados Coletados com Sucesso-----")
    display(dados_planilha)

    #Previsão do tempo
    print("-----Buscando Previsões do Tempo-----")

    # Iterar cada resposta do formulario
    for indice, linha in dados_planilha.iterrows():
        nome_local = linha.get("Nome do Local", linha.get("Nome do local", "Local Desconhecido"))

        # Limpar e corrigir as coordenadas usando a nova função
        lat = limpar_coordenada(linha.get("Latitude"))
        lon = limpar_coordenada(linha.get("Longitude"))

        if lat is None or lon is None:
            print(f"Local: {nome_local} - Coordenadas inválidas detectadas ({linha.get('Latitude')}, {linha.get('Longitude')})\n")
            continue

        parametros = {
            "latitude": lat,
            "longitude": lon,
            "current": "temperature_2m"
        }

        #Resposta
        resposta = requests.get(endpoint_api, params=parametros).json()
        print(f"Local: {nome_local} (Lat: {lat}, Lon: {lon})")

        # Verifica se a API retornou erro interno
        if "error" in resposta:
            print(f"Erro retornado pela API: {resposta.get('reason')}")
        else:
            display(resposta)

        print('\n')
        time.sleep(1) # Pausa de 1 segundo para respeitar limites da API

except Exception as e:
    print(f"Ocorreu um erro durante a execução: {e}")

-----Dados Coletados com Sucesso-----


,Carimbo de data/hora,Nome do local,Latitude,Longitude
0,18/11/2026 14:48:40,Estatua do bom Jesus,-24.7,-47.55
1,18/11/2026 15:34:21,Catedral da Se,-23.55,-46.63
2,18/11/2026 15:43:56,Campinas,"22,9","47,06"
3,18/11/2026 15:44:23,Jardim Itaguaçu,-23.046.753,-47.128.319


-----Buscando Previsões do Tempo-----
Local: Estatua do bom Jesus (Lat: -24.7, Lon: -47.55)


{'latitude': -24.710018,
 'longitude': -47.556274,
 'generationtime_ms': 0.054717063903808594,
 'utc_offset_seconds': 0,
 'timezone': 'GMT',
 'timezone_abbreviation': 'GMT',
 'elevation': 111.0,
 'current_units': {'time': 'iso8601',
  'interval': 'seconds',
  'temperature_2m': '°C'},
 'current': {'time': '2026-11-18T23:00',
  'interval': 900,
  'temperature_2m': 23.0}}



Local: Catedral da Se (Lat: -23.55, Lon: -46.63)


{'latitude': -23.514938,
 'longitude': -46.610504,
 'generationtime_ms': 0.043511390686035156,
 'utc_offset_seconds': 0,
 'timezone': 'GMT',
 'timezone_abbreviation': 'GMT',
 'elevation': 737.0,
 'current_units': {'time': 'iso8601',
  'interval': 'seconds',
  'temperature_2m': '°C'},
 'current': {'time': '2026-11-18T23:00',
  'interval': 900,
  'temperature_2m': 22.2}}



Local: Campinas (Lat: 22.9, Lon: 47.06)


{'latitude': 22.882248,
 'longitude': 47.017727,
 'generationtime_ms': 0.08988380432128906,
 'utc_offset_seconds': 0,
 'timezone': 'GMT',
 'timezone_abbreviation': 'GMT',
 'elevation': 644.0,
 'current_units': {'time': 'iso8601',
  'interval': 'seconds',
  'temperature_2m': '°C'},
 'current': {'time': '2026-11-18T23:00',
  'interval': 900,
  'temperature_2m': 15.5}}



Local: Jardim Itaguaçu (Lat: -23046.753, Lon: -47128.319)
Erro retornado pela API: Latitude must be in range of -90 to 90°. Given: -23046.754.


